In [15]:
import pandas as pd
import numpy as np

In [16]:
df=pd.read_csv(r"../../data/processed/merged_ieee.csv")
df.tail()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590539,3577539,0,15811131,279.95,W,15066,170.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Sorting relative time data in chronological order
df=df.sort_values('TransactionDT').reset_index(drop=True)

In [18]:
n=len(df)
train_end=int(0.8*n)
val_end=int(0.9*n)
df_train=df.iloc[:train_end].copy()
df_val=df.iloc[train_end:val_end].copy()
df_test=df.iloc[val_end:].copy()

In [19]:
print(f"Train: {len(df_train)} rows")
print(f"Fraud rate: {df_train['isFraud'].mean()}")
print(f"Validation: {len(df_val)} rows")
print(f"Fraud rate: {df_val['isFraud'].mean()}")
print(f"Train: {len(df_test)} rows")
print(f"Fraud rate: {df_test['isFraud'].mean()}")


Train: 472432 rows
Fraud rate: 0.03513521522674162
Validation: 59054 rows
Fraud rate: 0.03134419345006265
Train: 59054 rows
Fraud rate: 0.03747417617773563


In [20]:
#Save the splits to csvs

df_train.to_csv("../../data/processed/train.csv", index=False)
df_val.to_csv("../../data/processed/val.csv", index=False)
df_test.to_csv("../../data/processed/test.csv", index=False)

## Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
df_train=pd.read_csv(r"../../data/processed/train.csv")
df_val=pd.read_csv(r"../../data/processed/val.csv")
df_test=pd.read_csv(r"../../data/processed/test.csv")

In [39]:
df_train.shape

(472432, 434)

In [3]:
# Time features
# Note: Not real clock time, instead it is relative
# Model can learn patterns from time relative values so using them as features

def add_time_features(df):
    df=df.copy()
    df['hour']=df['TransactionDT']%86400//3600
    df['day_of_week']=(df['TransactionDT']//86400)%7
    return df

df_train=add_time_features(df_train)
df_val=add_time_features(df_val)
df_test=add_time_features(df_test)
print("Time features done")

Time features done


In [4]:
def add_amount_features(df,df_train=None):
    df=df.copy()
    df['log_amount']=np.log1p(df['TransactionAmt'])
    if df_train is not None:
        source=df_train
    else:
        source=df
    card_mean=source.groupby('card1')['TransactionAmt'].mean().rename('card1_mean')
    df=df.merge(card_mean,on='card1',how='left')
    df['card1_mean']=df['card1_mean'].fillna(df['TransactionAmt'].median())
    df['amount_to_card_mean']=df['TransactionAmt']/(df['card1_mean']+1e-8)
    df.drop(columns='card1_mean',inplace=True)
    return df

df_train=add_amount_features(df_train,df_train=None)
df_val=add_amount_features(df_val,df_train=df_train)
df_test=add_amount_features(df_test,df_train=df_train)
print("Amount features done")


Amount features done


In [5]:
def add_email_features(df, df_train=None):
    df = df.copy()
    # Purchaser and recepient mail mismatch signals identity inconsistency
    df['email_domain_mismatch'] = (
        df['P_emaildomain'].fillna('unknown') != df['R_emaildomain'].fillna('unknown')
    ).astype(int)
    
    # Free consumer domains -> Less identity verification
    high_risk_domains = ['gmail.com', 'yahoo.com', 'hotmail.com', 'anonymous.com']
    df['purchaser_email_risk'] = df['P_emaildomain'].isin(high_risk_domains).astype(int)
    
    # Frequency encoding is done instead of one-hot
    source = df_train if df_train is not None else df
    p_freq = source['P_emaildomain'].value_counts(normalize=True)
    df['p_email_freq'] = df['P_emaildomain'].map(p_freq).fillna(0)
    return df

df_train = add_email_features(df_train, df_train=None)
df_val   = add_email_features(df_val,   df_train=df_train)
df_test  = add_email_features(df_test,  df_train=df_train)
print("Email features done")

Email features done


In [6]:
def add_device_features(df):
    df = df.copy()
    df['is_mobile'] = (df['DeviceType'] == 'mobile').astype(int)
    df['has_identity']  = df['DeviceType'].notna().astype(int)
    return df

df_train = add_device_features(df_train)
df_val   = add_device_features(df_val)
df_test  = add_device_features(df_test)
print("Device features done")

Device features done


In [7]:
def add_card_features(df,df_train=None):
    df=df.copy()
    source=df_train if df_train is not None else df
    
    # card4 = network (visa/mastercard), card6 = type (debit/credit/prepaid)
    # card2, card3, card5 = additional card metadata — frequency encoded
    # Frequency encoding: rare card types carry more uncertainty
    for col in ['card2', 'card3', 'card4', 'card5', 'card6']:
        freq=source[col].value_counts(normalize=True)
        df[f'{col}_freq'] = df[col].map(freq).fillna(0)
    return df

df_train=add_card_features(df_train,df_train=None)
df_val=add_card_features(df_val,df_train=df_train)
df_test=add_card_features(df_test,df_train=df_train)
print("Card features done")

Card features done


In [8]:
def add_uid_features(df_train,df_val,df_test):
    for df in [df_train,df_val,df_test]:
        df['uid'] = (df['card1'].astype(str) + '_' +df['card2'].astype(str) + '_' +df['addr1'].astype(str))
        uid_count=df_train.groupby('uid')['isFraud'].count()
        uid_amt_mean=df_train.groupby('uid')['TransactionAmt'].mean()
        uid_amt_std=df_train.groupby('uid')['TransactionAmt'].std().fillna(0)
        card1_amt_mean=df_train.groupby('card1')['TransactionAmt'].mean()

        global_amt_mean=df_train['TransactionAmt'].mean()
        global_amt_std=df_train['TransactionAmt'].std()
    for df in [df_train, df_val, df_test]:
        df['uid_count']=df['uid'].map(uid_count).fillna(0)
        df['uid_amt_mean']=df['uid'].map(uid_amt_mean).fillna(global_amt_mean)
        df['uid_amt_std']=df['uid'].map(uid_amt_std).fillna(global_amt_std)
        df['uid_amt_zscore']=((df['TransactionAmt']-df['uid_amt_mean'])/df['uid_amt_std'].replace(0, 1))
        df['card1_amt_mean']=df['card1'].map(card1_amt_mean).fillna(global_amt_mean)
        df['amt_to_card1_mean']=df['TransactionAmt']/df['card1_amt_mean'].replace(0,1)
    return df_train,df_val,df_test

df_train, df_val, df_test=add_uid_features(df_train,df_val,df_test)
print("UID features done")



UID features done


In [9]:
def add_product_features(df_train, df_val, df_test):
    prod_train=pd.get_dummies(df_train['ProductCD'], prefix='prod')
    prod_val=pd.get_dummies(df_val['ProductCD'],   prefix='prod')
    prod_test=pd.get_dummies(df_test['ProductCD'],  prefix='prod')

    # Align columns becoz val/test may be missing some ProductCD values
    for col in prod_train.columns:
        if col not in prod_val.columns:  prod_val[col]  = 0
        if col not in prod_test.columns: prod_test[col] = 0
    prod_val=prod_val[prod_train.columns]
    prod_test=prod_test[prod_train.columns]

    df_train=pd.concat([df_train, prod_train], axis=1)
    df_val=pd.concat([df_val,   prod_val],   axis=1)
    df_test=pd.concat([df_test,  prod_test],  axis=1)
    return df_train, df_val, df_test

df_train,df_val,df_test=add_product_features(df_train,df_val,df_test)
print("Product features done")

Product features done


In [10]:
# C columns
c_cols=[f'C{i}' for i in range(1, 15) if f'C{i}' in df_train.columns]

# D columns
d_cols=[f'D{i}' for i in range(1, 16) if f'D{i}' in df_train.columns]

# M columns
m_cols=[f'M{i}' for i in range(1, 10) if f'M{i}' in df_train.columns]

# M columns are T/F/NaN so we basically encode as 1/0/-1
for df in [df_train, df_val, df_test]:
    for col in m_cols:
        df[col] = df[col].map({'T':1,'F':0}).fillna(-1)

# dist columns
dist_cols=[c for c in ['dist1', 'dist2'] if c in df_train.columns]

# addr columns
addr_cols=[c for c in ['addr1', 'addr2'] if c in df_train.columns]

# ID columns
numeric_id_cols=[c for c in df_train.columns if c.startswith('id_') and df_train[c].dtype in ['float64', 'int64']
]

# Categorical id columns — frequency encoded from train only
cat_id_cols = [
    c for c in df_train.columns
    if c.startswith('id_') and df_train[c].dtype == 'object'
]
for col in cat_id_cols:
    freq = df_train[col].value_counts(normalize=True)
    df_train[f'{col}_freq']=df_train[col].map(freq).fillna(0)
    df_val[f'{col}_freq']=df_val[col].map(freq).fillna(0)
    df_test[f'{col}_freq']=df_test[col].map(freq).fillna(0)

cat_id_freq_cols=[f'{c}_freq' for c in cat_id_cols]
prod_cols=[c for c in df_train.columns if c.startswith('prod_')]

print(f"C cols: {len(c_cols)}, D cols: {len(d_cols)}, M cols: {len(m_cols)}")
print(f"numeric id cols: {len(numeric_id_cols)}, cat id freq cols: {len(cat_id_freq_cols)}")


C cols: 14, D cols: 15, M cols: 9
numeric id cols: 23, cat id freq cols: 15


In [11]:
FEATURE_COLS = (
    # Time
    ['hour', 'day_of_week'] +
    # Amount
    ['log_amount', 'amount_to_card_mean'] +
    # Email
    ['email_domain_mismatch', 'purchaser_email_risk', 'p_email_freq'] +
    # Device
    ['is_mobile','has_identity'] +
    # Card — frequency encoded
    ['card2_freq', 'card3_freq', 'card4_freq', 'card5_freq', 'card6_freq'] +
    # UID behavioural aggregations
    ['uid_count', 'uid_amt_mean', 'uid_amt_std', 'uid_amt_zscore',
     'card1_amt_mean', 'amt_to_card1_mean']+
     prod_cols+c_cols+d_cols+m_cols+dist_cols+addr_cols+numeric_id_cols+cat_id_freq_cols
)

# Remove duplicates preserving order
seen=set()
FEATURE_COLS=[c for c in FEATURE_COLS if not (c in seen or seen.add(c))]

print(f"Total features: {len(FEATURE_COLS)}")

Total features: 105


In [12]:
X_train=df_train[FEATURE_COLS].fillna(-999)
y_train=df_train['isFraud']

X_val=df_val[FEATURE_COLS].fillna(-999)
y_val=df_val['isFraud']

X_test=df_test[FEATURE_COLS].fillna(-999)
y_test=df_test['isFraud']

print(f"Feature matrix: {X_train.shape}")
print(f"Fraud in train: {y_train.sum()}")

Feature matrix: (472432, 105)
Fraud in train: 16599


In [13]:
df_train.to_csv("../../data/processed/train_engineered.csv", index=False)
df_val.to_csv("../../data/processed/val_engineered.csv",     index=False)
df_test.to_csv("../../data/processed/test_engineered.csv",   index=False)

joblib.dump(FEATURE_COLS, "../../data/processed/feature_cols.pkl")
print(f"Saved. Feature count: {len(FEATURE_COLS)}")

Saved. Feature count: 105
